# Data Exploration

Use this notebook to explore the raw data before building your dbt models.

**Make sure you've run `uv run python scripts/init_db.py` first.**

In [ ]:
import duckdb

conn = duckdb.connect('../mock_data.duckdb', read_only=True)
print('Connected!')

## Raw Tables Overview

In [ ]:
# Row counts
for table in ['users', 'courses', 'enrolments', 'events']:
    count = conn.sql(f'SELECT COUNT(*) AS n FROM raw.{table}').fetchone()[0]
    print(f'raw.{table}: {count} rows')

## Users

In [ ]:
conn.sql('SELECT * FROM raw.users LIMIT 10')

In [ ]:
# Check for duplicates and deleted users
conn.sql("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_users,
        COUNT(*) FILTER (WHERE deleted = TRUE) AS deleted_users,
        MIN(signupDate) AS earliest_signup,
        MAX(signupDate) AS latest_signup
    FROM raw.users
""")

## Courses

In [ ]:
conn.sql('SELECT * FROM raw.courses LIMIT 10')

## Enrolments

In [ ]:
conn.sql("""
    SELECT 
        COUNT(*) AS total_enrolments,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT course_id) AS unique_courses,
        status, COUNT(*) AS n
    FROM raw.enrolments
    GROUP BY status
""")

## Events

In [ ]:
conn.sql('SELECT * FROM raw.events LIMIT 10')

In [ ]:
# Event type distribution
conn.sql("""
    SELECT event_type, COUNT(*) AS n
    FROM raw.events
    GROUP BY event_type
    ORDER BY n DESC
""")

In [ ]:
# Daily event volume
conn.sql("""
    SELECT 
        event_timestamp::DATE AS event_date,
        COUNT(*) AS n_events,
        COUNT(DISTINCT user_id) AS n_users
    FROM raw.events
    GROUP BY event_date
    ORDER BY event_date
""")

In [ ]:
# Date range and coverage
conn.sql("""
    SELECT 
        COUNT(*) AS total_events,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT course_id) AS unique_courses,
        MIN(event_timestamp::DATE) AS first_event,
        MAX(event_timestamp::DATE) AS last_event,
        COUNT(DISTINCT event_timestamp::DATE) AS active_days
    FROM raw.events
""")

In [ ]:
conn.close()